In [2]:
import openslide
from openslide import open_slide
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
import cv2 as cv
import os
import glob

### Split into Patches and then save as .png images. 

#### This takes in a magnification rescaled image based on the BIT microscope image and Hamamatsu slide scanner, and then splits the image into n x n patches. 

In [3]:
import os
import random
import shutil
from glob import glob

def split_image_into_patches(image, patch_size, overlap, output_folder, handle="patch", extension="png", random_samples=0, random_seed=None):
    """
    Splits an image into n x n patches with overlap, and saves them to a folder.
    Only saves complete patches of the specified size.
    Args:
    - image: PIL image to split
    - patch_size: Size of the patches (e.g., (256, 256))
    - overlap: Overlap between patches (e.g., 50 for 50-pixel overlap)
    - output_folder: Folder to save the patches
    - handle: Prefix for the patch file name
    - extension: Output image extension (e.g., "png", "tif")
    - random_samples: Number of randomly sampled patches to save
    - random_seed: Seed for reproducible random sampling
    Returns:
    - None
    """
    # Ensure the output folder exists
    os.makedirs(output_folder, exist_ok=True)
    # Convert the image to a NumPy array
    np_image = np.array(image)
    # Get the dimensions of the image
    height, width = np_image.shape[0], np_image.shape[1]
    # Calculate the step size (patch size minus overlap)
    step = patch_size[0] - overlap
    patch_number = 1  # Counter for patches
    # Iterate through the image and extract patches
    for y in range(0, height - patch_size[0] + 1, step):
        for x in range(0, width - patch_size[1] + 1, step):
            # Check if the patch is within the image boundaries
            if y + patch_size[0] <= height and x + patch_size[1] <= width:
                # Extract the patch
                patch = np_image[y:y + patch_size[0], x:x + patch_size[1]]
                # Convert the patch to a PIL image
                patch_pil = Image.fromarray(patch)
                # Save the patch
                formatted = f"{patch_number:05d}"
                patch_filename = f"{handle}_{formatted}.{extension}"
                patch_pil.save(os.path.join(output_folder, patch_filename))
                patch_number += 1

    # Randomly sample additional patches (uniform across valid coordinates)
    if random_samples and random_samples > 0:
        rng = np.random.default_rng(random_seed)
        random_number = 1
        max_y = height - patch_size[0]
        max_x = width - patch_size[1]
        if max_y >= 0 and max_x >= 0:
            for _ in range(random_samples):
                y = int(rng.integers(0, max_y + 1))
                x = int(rng.integers(0, max_x + 1))
                patch = np_image[y:y + patch_size[0], x:x + patch_size[1]]
                patch_pil = Image.fromarray(patch)
                formatted = f"{random_number:05d}"
                patch_filename = f"{handle}_rand_{formatted}.{extension}"
                patch_pil.save(os.path.join(output_folder, patch_filename))
                random_number += 1


def split_images_train_test(source_folder, output_root, train_ratio=0.99, seed=42):
    """
    Randomly splits .tif images in source_folder into train and test folders.

    Args:
        source_folder (str): Path to folder containing .tif images.
        output_root (str): Root folder where 'train/' and 'test/' will be created.
        train_ratio (float): Ratio of images to put in train set.
        seed (int): Random seed for reproducibility.
    """
    os.makedirs(os.path.join(output_root, 'train'), exist_ok=True)
    os.makedirs(os.path.join(output_root, 'test'), exist_ok=True)

    tif_images = glob(os.path.join(source_folder, "*.tif"))
    random.seed(seed)
    random.shuffle(tif_images)

    split_index = int(len(tif_images) * train_ratio)
    train_images = tif_images[:split_index]
    test_images = tif_images[split_index:]

    print(f"Total images: {len(tif_images)}")
    print(f"Train: {len(train_images)}")
    print(f"Test: {len(test_images)}")

    for img_path in train_images:
        shutil.move(img_path, os.path.join(output_root, 'train', os.path.basename(img_path)))

    for img_path in test_images:
        shutil.move(img_path, os.path.join(output_root, 'test', os.path.basename(img_path)))

# Example usage:
# split_images_train_test("/path/to/images", "/path/to/split_output")


In [4]:
# Magnification Information of the microscope and slide scanner. 

mpp_x = float(0.22034197073858627)
mpp_y = float(0.22034197073858627)
print(f"MPP X: {mpp_x} µm/pixel")
print(f"MPP Y: {mpp_y} µm/pixel")
PCO_pixel_size = 6.5 # microns
BIT_magnification = 40
reference_scale = PCO_pixel_size / BIT_magnification
scale_factor_x = mpp_x / reference_scale
scale_factor_y = mpp_x / reference_scale

print(f"x ratio: {scale_factor_x}")
print(f"y ratio: {scale_factor_y}")

MPP X: 0.22034197073858627 µm/pixel
MPP Y: 0.22034197073858627 µm/pixel
x ratio: 1.3559505891605308
y ratio: 1.3559505891605308


In [ ]:
slide_paths = r'C:\Users\durrlab-asong\Desktop\virtual_staining_training_data\duodenum\FFPE-HE\duodenum_crypts_online\cropped_to_train'
imgs = glob(os.path.join(slide_paths, "*.jpg"))
print(len(imgs))
patch_size = (512, 512)

patch_handle = "duodenum_crypts_lieberkuhn"
overlap = 400 # number of pixel overlap
img_ext = "tif"
n_random_samples = 100
largest_image = 2567 * 1932
random_seed = 42

# Account for shrinkage during dehydration
percent_shrinkage = 1  # 50% shrinkage means the final image is 50% of the original size 
shrinkage_compensation = 1 / percent_shrinkage  # 3.33x magnification

# Convert percent_shrinkage to string format (replace . with p)
shrinkage_str = str(percent_shrinkage).replace('.', 'p')
output_slide_path = os.path.join(slide_paths, f"patches_shrinkage_compensation={shrinkage_str}")

# Combine with existing scale factors
final_scale_x = scale_factor_x * shrinkage_compensation
final_scale_y = scale_factor_y * shrinkage_compensation

for i, img in enumerate(imgs):
    img_roi = Image.open(img)

    # Rescale image. 
    image_roi = img_roi.convert('RGB')
    image_roi = np.array(image_roi)
    n_random_samples_effective = round(n_random_samples * (image_roi.shape[0] * image_roi.shape[1]) / largest_image)

    # rescale H&E Image for magnification of microscope and slide scanner and Rescale H&E image for shrinkage compensation during dehydration.
    rescaled_image_roi = cv.resize(image_roi, None, 
                               fx=final_scale_x, 
                               fy=final_scale_y, 
                               interpolation=cv.INTER_CUBIC)

    print(f"Rescaled image shape: {rescaled_image_roi.shape} and Original image shape: {image_roi.shape}")
    rescaled_image_roi = Image.fromarray(rescaled_image_roi)
    patch_handle_temp = f"{patch_handle}_roi_{i}"
    split_image_into_patches(rescaled_image_roi, patch_size, overlap, output_slide_path, handle=patch_handle_temp, extension=img_ext, random_samples=n_random_samples_effective, random_seed=random_seed)

4
Rescaled image shape: (2620, 3493, 3) and Original image shape: (1932, 2576, 3)
Rescaled image shape: (2620, 3493, 3) and Original image shape: (1932, 2576, 3)
Rescaled image shape: (2620, 3493, 3) and Original image shape: (1932, 2576, 3)
Rescaled image shape: (2620, 3493, 3) and Original image shape: (1932, 2576, 3)


In [7]:
split_images_train_test(output_slide_path, output_slide_path, train_ratio=0.99, seed=42)

Total images: 1080
Train: 1069
Test: 11
